In [1]:
import transformers, peft, torch, numpy as np, librosa, pathlib, re, warnings, json, gc, time
from pprint import pprint
from tqdm import tqdm
from datasets import Dataset

In [2]:
warnings.filterwarnings("ignore", category=UserWarning, module="librosa")
warnings.filterwarnings("ignore", category=FutureWarning, module="librosa")

warnings.filterwarnings("ignore", category=UserWarning, module="transformers")
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers")

In [3]:
model_name = "Qwen/Qwen3-0.6B"
torch_dtype = torch.float16

In [4]:
sys_prompt_short = """Ты помощник лектора. Прочитай конспект и составь краткий план лекции 
    в виде 3–5 пунктов. Не пиши пояснений и не используй формулы. Отвечай только на русском. 
    Игнорируй не по теме: политику, мат, вопросы студентов. 
"""

In [6]:
with open("dataset_checkpoint.json", "r", encoding = "utf-8") as file:
    raw_dataset = json.load(file)

formatted_data = []

for item in raw_dataset:
    formatted_data.append({
        "text" : sys_prompt_short + item['text'],
        "gpt_plan" : item['plan'],
    })

hf_dataset = Dataset.from_list(formatted_data)

In [7]:
bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
    low_cpu_mem_usage=True
)

In [8]:
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
  #  torch_dtype=torch_dtype,
    quantization_config=bnb_config,
    device_map="auto",
    use_cache=False,
)

In [9]:
def format_example(item):
    input_text = item['text'] 
    target_text = item['gpt_plan'] + tokenizer.eos_token
    full_text = input_text + target_text

    tokenized = tokenizer(
        full_text,
        truncation=True,
        padding=False
    )

    # прописываем, чтобы модель не училась предсказывать system_srompt + prompt
    input_len = len(tokenizer(input_text, truncation=True, padding=False)['input_ids'])
    labels = [-100] * input_len + tokenized['input_ids'][input_len:]

    tokenized['labels'] = labels
    return tokenized

dataset_tokenized = hf_dataset.map(format_example, remove_columns=["text", "gpt_plan"])

Map:   0%|          | 0/284 [00:00<?, ? examples/s]

In [10]:
lora_config = peft.LoraConfig(
    r = 8,
    lora_alpha=16,
    lora_dropout = 0.1,
    target_modules=["q_proj", "v_proj"],
    task_type=peft.TaskType.CAUSAL_LM,
)

In [11]:
print(len(dataset_tokenized['input_ids'][0]))

13836


In [12]:
model = peft.prepare_model_for_kbit_training(model)
model = peft.get_peft_model(model, lora_config)

In [13]:
trainer = transformers.Trainer(
    model=model, train_dataset=dataset_tokenized, 
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1, gradient_accumulation_steps=1,
        warmup_steps=250, num_train_epochs=6, learning_rate=2e-4, fp16=True,
        logging_steps=1, output_dir='outputs',  gradient_checkpointing=True,
        optim="paged_adamw_8bit", 
        ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [14]:
print(f"Percentage of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad) / sum(p.numel() for p in model.parameters())}")
allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3
print(f"GPU Memory Allocated: {allocated:.2f} GB")
print(f"GPU Memory Reserved:  {reserved:.2f} GB")

Percentage of trainable parameters: 0.003042155584528466
GPU Memory Allocated: 0.80 GB
GPU Memory Reserved:  1.67 GB


In [15]:
trainer.train()

OutOfMemoryError: CUDA out of memory. Tried to allocate 8.90 GiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Of the allocated memory 7.38 GiB is allocated by PyTorch, and 215.68 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)